# Add Capacity to Stations Features

This notebook:
- Loads `stations_features.csv`
- Loads `dublin_bikes_all.parquet`
- Extracts station capacity
- Checks consistency (capacity per station)
- Saves `stations_features_v2.parquet`


In [20]:
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ALL_PATH = PROCESSED_DIR / 'dublin_bikes_all.parquet'
if not ALL_PATH.exists():
    ALL_PATH = PROCESSED_DIR / 'dublin_bikes_all.csv.gz'

if ALL_PATH.suffix == '.parquet':
    df = pd.read_parquet(ALL_PATH)
else:
    df = pd.read_csv(ALL_PATH)

df['last_reported'] = pd.to_datetime(df['last_reported'], errors='coerce')
df = df.sort_values(['station_id', 'last_reported']).copy()
print(df.shape)


# --- Paths ---
stations_path = PROCESSED_DIR / 'stations_features.parquet'
raw_path = ALL_PATH
output_path = PROCESSED_DIR / 'stations_features_v2.parquet'

stations = pd.read_parquet(stations_path)
raw = pd.read_parquet(raw_path)

print(stations.head())
print(raw.head())


(13513883, 9)
   station_id        lat       lon  distance_center  center_index  \
0           1  53.340919 -6.262501         0.009149             1   
1           2  53.356781 -6.268140         0.010497             2   
2           3  53.351185 -6.269859         0.009658             1   
3           4  53.346886 -6.272977         0.013008             2   
4           5  53.330673 -6.260178         0.019127             2   

  area_type_str  tourism  
0        retail     True  
1   residential     True  
2   residential     True  
3   residential     True  
4   residential     True  
        last_reported  station_id  num_bikes_available  num_docks_available  \
0 2022-01-01 00:00:04           2                   10                   10   
1 2022-01-01 00:00:04           3                    1                   19   
2 2022-01-01 00:00:04           4                   11                    9   
3 2022-01-01 00:00:04           5                   23                   17   
4 2022-01-01 0

In [21]:
# --- Detect columns ---
station_candidates = ["station_id", "stationid", "number", "id"]
capacity_candidates = ["capacity", "bike_stands", "station_capacity", "total_stands"]

station_col = next(c for c in station_candidates if c in raw.columns)
capacity_col = next(c for c in capacity_candidates if c in raw.columns)

print("Station column:", station_col)
print("Capacity column:", capacity_col)


Station column: station_id
Capacity column: capacity


In [22]:
# --- Sort once ---
raw_sorted = raw.sort_values(["station_id", "last_reported"])

# --- Detect changes ---
raw_sorted["capacity_prev"] = raw_sorted.groupby("station_id")[capacity_col].shift()
raw_sorted["capacity_changed"] = raw_sorted[capacity_col] != raw_sorted["capacity_prev"]

# ignore first row per station
raw_sorted["capacity_changed"] &= raw_sorted["capacity_prev"].notna()

# --- Extract change timestamps (first + last change) ---
change_events = raw_sorted[raw_sorted["capacity_changed"]].copy()

change_summary = (
    change_events
    .groupby("station_id")
    .agg(
        first_capacity_change=("last_reported", "min"),
        last_capacity_change=("last_reported", "max"),
    )
    .reset_index()
)

# --- Latest capacity ---
latest_capacity = (
    raw_sorted
    .dropna(subset=[capacity_col])
    .groupby("station_id")
    .tail(1)[["station_id", capacity_col]]
    .rename(columns={capacity_col: "capacity"})
)

# --- Count unique capacities ---
capacity_counts = (
    raw_sorted
    .groupby("station_id")[capacity_col]
    .nunique()
    .reset_index(name="n_unique_capacity")
)

# --- Merge everything ---
capacity_lookup = (
    latest_capacity
    .merge(capacity_counts, on="station_id")
    .merge(change_summary, on="station_id", how="left")
)

capacity_lookup["capacity_non_unique"] = capacity_lookup["n_unique_capacity"] > 1

# --- Inspect ---
print(capacity_lookup[capacity_lookup["capacity_non_unique"]])

    station_id  capacity  n_unique_capacity first_capacity_change  \
68          70        21                  2   2025-05-07 11:15:00   

   last_capacity_change  capacity_non_unique  
68  2025-05-07 11:15:00                 True  


In [23]:
# --- Merge into stations_features ---
stations_v2 = stations.merge(capacity_lookup, on="station_id", how="left")


In [24]:
# --- Save output ---
stations_v2.to_parquet(output_path, index=False)
print("Saved:", output_path)


Saved: /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/stations_features_v2.parquet
